In [27]:
import gurobipy as gp
from gurobipy import GRB
from utils.data import *
import math

In [89]:
start= dt.datetime(year=2000, month=1, day=1)
end= dt.datetime(year=2021, month=1, day=1)
all_stocks = ["NVDA", "AAPL", 'GOOG', "AVGO", "WMT", "JPM", "LLY", "XOM", "KO"]

n = len(all_stocks)

pce = load_pce()
effr = load_ffr()

a = (0.0001, 0)

mu = generate_mu(all_stocks, start, end)
Sigma = generate_Sigma(all_stocks, start, end)

new_sigma, sep_Sigma = add_covariates_to_covar(Sigma, all_stocks, [pce, effr], start, end)
new_mu, sep_mu = add_covariates_to_mu(mu, [pce, effr])

mu, Sigma = conditional_moments(sep_mu, sep_Sigma, a)

r = (1+a[1])**(1/4)-1
T = 1/4
S = [int(yf.Ticker(s).history(period="1d").Open) for s in all_stocks]

/var/folders/x_/mtr88x3523j2pp7lgmpl_7k40000gp/T/ipykernel_19143/1380804487.py:22: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  S = [int(yf.Ticker(s).history(period="1d").Open) for s in all_stocks]
/var/folders/x_/mtr88x3523j2pp7lgmpl_7k40000gp/T/ipykernel_19143/1380804487.py:22: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  S = [int(yf.Ticker(s).history(period="1d").Open) for s in all_stocks]
/var/folders/x_/mtr88x3523j2pp7lgmpl_7k40000gp/T/ipykernel_19143/1380804487.py:22: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  S = [int(yf.Ticker(s).history(period="1d").Open) for s in all_stocks]
/var/folders/x_/mtr88x3523j2pp7lgmpl_7k40000gp/T/ipykernel_19143/1380804487.py:22: FutureWarning: Calling int on a sin

In [90]:
S

[199, 272, 337, 424, 131, 311, 900, 149, 76]

In [104]:
m = gp.Model()

m.setParam("MIPGap", 0.01)

S = [yf.Ticker(s).history(period="1d").Close.to_list()[0] for s in all_stocks]
sigma = [np.sqrt(Sigma[i, i]) for i in range(n)]

w = m.addMVar(n, lb=0, name="weight")

mu_d = m.addMVar(n, lb=-GRB.INFINITY, name="expectation")
Delta = m.addMVar(n, lb=0.1, ub=.5, name="Delta")
#Phi_inv = m.addMVar(n, lb=-GRB.INFINITY, name="phi of inverse CDF")
#phi = m.addMVar(n, lb=-GRB.INFINITY, name="phi")
#K = m.addMVar(n, lb=0, name="strike")
P = m.addMVar(n, lb=-GRB.INFINITY, name="put price")
#d1 = m.addMVar(n, lb=-GRB.INFINITY, name="d1")
#d2 = m.addMVar(n, lb=-GRB.INFINITY, name="d2")
var_norm = m.addVar()

x_pts_CDF = np.linspace(-5, 5, 101)
y_pts_CDF = np.array([math.erf(x/np.sqrt(2))*1/2+1/2 for x in x_pts_CDF])

x_pts_PDF = np.linspace(-5, 5, 101)
y_pts_PDF = np.array([math.exp(-x**2/2)*1/np.sqrt(2*np.pi) for x in x_pts_CDF])

for i in range(n):
    Phi_inv_expr = 4/1.7*(Delta[i] - 0.5)
    
    K_expr = (sigma[i]*Phi_inv_expr + mu[i] + 1)*S[i]
    

    d1 = m.addVar(lb=-GRB.INFINITY, name=f"d1_{i}")
    d2 = m.addVar(lb=-GRB.INFINITY, name=f"d2_{i}")

    m.addConstr(d1 == (1 - K_expr/S[i] + T*(r + sigma[i]**2/2))/(sigma[i]*np.sqrt(T)))
    m.addConstr(d2 == d1 - sigma[i]*np.sqrt(T))

    cdf_d1_expr = m.addVar(lb=-GRB.INFINITY, name=f"cdf_d1_{i}")
    cdf_d2_expr = m.addVar(lb=-GRB.INFINITY, name=f"cdf_d2_{i}")

    cdf_d1 = m.addGenConstrPWL(d1, cdf_d1_expr, x_pts_CDF,\
                                                y_pts_CDF)
    cdf_d2 = m.addGenConstrPWL(d2, cdf_d2_expr, x_pts_CDF,\
                                                y_pts_CDF)

    d1_expr = (1 - K_expr/S[i] + T*(r + sigma[i]**2/2))/(sigma[i]*np.sqrt(T))
    d2_expr =  d1_expr - sigma[i]*np.sqrt(T)
    cdf_d1_expr = (0.5 - (1/np.sqrt(2*np.pi))*d1_expr)
    cdf_d2_expr = 0.5 - (1/np.sqrt(2*np.pi))*d2_expr
    
    
    Phi_inv = m.addVar(lb=-GRB.INFINITY, name=f"phi_inv_{i}")
    phi = m.addVar(lb=-GRB.INFINITY, name=f"phi_{i}")

    m.addConstr(Phi_inv == Phi_inv_expr)
    
    phi_i = m.addGenConstrPWL(Phi_inv, phi, x_pts_PDF,\
                                            y_pts_PDF)

    P_expr = K_expr*np.exp(-r*T)*cdf_d2_expr - S[i]*cdf_d1_expr

    m.addConstr(P[i] == P_expr)
    m.addConstr(mu_d[i] == mu[i] + sigma[i]*(Delta[i]*Phi_inv_expr+ phi) - P[i]/S[i])

t = m.addVar(lb=-GRB.INFINITY)

m.addConstr(gp.quicksum([wi for wi in w]) == 1)
m.addConstr(gp.quicksum([mu_d[i]*w[i] for i in range(n)]) >= var_norm * t)

Sigma_d = np.zeros((n, n), dtype=gp.Var)

for i in range(n):
    a_i = 4/1.7*(Delta[i] - 0.5)
    cdf_eta_i = m.addVar(lb=-GRB.INFINITY, name=f"cdf_eta_{i}")

    eta_i = m.addVar(lb=-GRB.INFINITY, name=f"eta_{i}")

    m.addConstr(eta_i == mu[i]/sigma[i]-a_i)

    cdf_etai = m.addGenConstrPWL(eta_i, cdf_eta_i, x_pts_CDF,\
                                                y_pts_CDF)
    phi_eta_i = m.addVar(lb=-GRB.INFINITY, name=f"phi_{i}")

    pdf_eta_i = m.addGenConstrPWL(eta_i, phi_eta_i, x_pts_PDF,\
                                            y_pts_PDF)
    
    for j in range(i+1, n):
        rho_ij = Sigma[i,j] / (sigma[i]*sigma[j])
        sqrt_1mrho2 = np.sqrt(1 - rho_ij**2)

        a_j = 4/1.7*(Delta[j] - 0.5)

        cdf_eta_j = m.addVar(lb=-GRB.INFINITY, name=f"cdf_eta_{i}")

        eta_j = m.addVar(lb=-GRB.INFINITY, name=f"eta_{i}")

        m.addConstr(eta_j == mu[j]/sigma[j]-a_j)

        
        cdf_etaj = m.addGenConstrPWL(eta_j, cdf_eta_j, x_pts_CDF,\
                                                    y_pts_CDF)
        
        phi_eta_j = m.addVar(lb=-GRB.INFINITY, name=f"phi_{j}")

       
        pdf_eta_j = m.addGenConstrPWL(eta_j, phi_eta_j, x_pts_PDF,\
                                                y_pts_PDF)

        w_ij = (eta_i - rho_ij*eta_j) / sqrt_1mrho2
        w_ji = (eta_j - rho_ij*eta_i) / sqrt_1mrho2

        Phi_w_ij = 0.5 + 1/np.sqrt(2*np.pi)*w_ij
        Phi_w_ji = 0.5 + 1/np.sqrt(2*np.pi)*w_ji

        Phi2 = cdf_eta_i*cdf_eta_j + rho_ij*phi_eta_i*phi_eta_j

        phi2 = phi_eta_i*phi_eta_j / sqrt_1mrho2        

       # lhs = m.addVar(lb=-GRB.INFINITY, name=f"EZZ_{i}_{j}_lhs")
        covar_ij = m.addVar(lb=-GRB.INFINITY, name=f"covar_{i}_{j}")

        #m.addConstr(lhs == EZZ_ij * Phi2)
        print(rho_ij)
        m.addConstr(
            covar_ij == mu[i]*mu[j]+(rho_ij
            + (rho_ij * a_i * phi_eta_i * Phi_w_ji
            + rho_ij * a_j * phi_eta_j * Phi_w_ij
            + (1 - rho_ij**2) * phi2))*sigma[i]*sigma[j]/Phi2 -  mu_d[i]*mu_d[j]
        )

        Sigma_d[i, j] = covar_ij      
        Sigma_d[j, i] = covar_ij
 
    
    var_i = m.addVar(lb=-GRB.INFINITY, name=f"var_{i}")

    m.addConstr(
        var_i == sigma[i]**2 *                                  \
        (1-phi_eta_i**2/(cdf_eta_i)**2 - eta_i*phi_eta_i/(cdf_eta_i))
    )

    Sigma_d[i, i] = var_i


m.addConstr(var_norm == gp.quicksum([Sigma_d[i, i] * w[i]**2 for i in range(n)]) + \
                        2*gp.quicksum([Sigma_d[i, j] * w[i]*w[j] for i in range(n) for j in range(n) if j > i]))

#m.setObjective(gp.quicksum(mu_d[i]*w[i] for i in range(n)), GRB.MAXIMIZE)
m.setObjective(t, GRB.MAXIMIZE)
#m.setObjective(var_norm, GRB.MINIMIZE)

m.optimize()


#m.addConstrs(Phi_inv[i] == -1/1.7*nlfunc.log(1/Delta[i]-1) for i in range(n))
"""
m.addConstrs(Phi_inv[i] == 4/1.7*(Delta[i]-1/2) for i in range(n))

m.addConstrs(phi[i] == 1/np.sqrt(2*np.pi)*(1-Phi_inv[i]**2/2) for i in range(n))
m.addConstrs(K[i] == sigma[i]*Phi_inv[i] + S[i] for i in range(n))

m.addConstrs(d1[i] == (1-K[i]/S[i]+T*(r+sigma[i]**2/2))/(sigma[i]*np.sqrt(T)) for i in range(n))
m.addConstrs(d2[i] == d1[i] - sigma[i]*np.sqrt(T) for i in range(n))

m.addConstrs(cdf_d1[i] == 0.5 - (1/np.sqrt(2*np.pi))*d1[i] for i in range(n))
m.addConstrs(cdf_d2[i] == 0.5 - (1/np.sqrt(2*np.pi))*d2[i] for i in range(n))

m.addConstrs(P[i] == K[i]*np.exp(-r*T)*cdf_d2[i] - S[i]*cdf_d1[i] for i in range(n))

"""


Set parameter MIPGap to value 0.01
0.5071064409915226
0.3128888636469143
0.1368884572658542
0.027119583312375836
0.3211627092363516
0.12314111072316959
0.22517385033330378
0.02364336827304721
0.3895614299086095
0.3905388536262933
0.09015621107624533
0.304378105470486
-0.05021918266582694
0.21206618377265044
-0.008719897379647951
0.07649396135894308
0.0875703306691407
0.19729478656081753
0.15772895864983444
0.2323310442951522
0.1866591142557546
0.09867753477483779
0.38161023942892913
0.16049198784244018
0.46575778581866045
0.20418540670042024
0.19719835868086164
0.1633661906839577
0.12738586841859606
0.35457467440665114
0.09439878637598585
0.3516612428092391
0.24482151973710448
0.1324792508992403
0.367746768669532
0.30636787765279583
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 24.0.0 24A348)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  0.01

Optimize a model with 73 rows, 272 c

'\nm.addConstrs(Phi_inv[i] == 4/1.7*(Delta[i]-1/2) for i in range(n))\n\nm.addConstrs(phi[i] == 1/np.sqrt(2*np.pi)*(1-Phi_inv[i]**2/2) for i in range(n))\nm.addConstrs(K[i] == sigma[i]*Phi_inv[i] + S[i] for i in range(n))\n\nm.addConstrs(d1[i] == (1-K[i]/S[i]+T*(r+sigma[i]**2/2))/(sigma[i]*np.sqrt(T)) for i in range(n))\nm.addConstrs(d2[i] == d1[i] - sigma[i]*np.sqrt(T) for i in range(n))\n\nm.addConstrs(cdf_d1[i] == 0.5 - (1/np.sqrt(2*np.pi))*d1[i] for i in range(n))\nm.addConstrs(cdf_d2[i] == 0.5 - (1/np.sqrt(2*np.pi))*d2[i] for i in range(n))\n\nm.addConstrs(P[i] == K[i]*np.exp(-r*T)*cdf_d2[i] - S[i]*cdf_d1[i] for i in range(n))\n\n'

In [105]:
weights = np.zeros(n)

var = m.getVars()

weights = [v.X for v in var if "weight" in v.VarName]

deltas = [v.X for v in var if "Delta" in v.VarName]
P = [v.X for v in var if "price" in v.VarName]
d1 = [v.X for v in var if v.VarName[:2] == "d1"]
d2 = [v.X for v in var if v.VarName[:2] == "d2"]
cdf_d1 = [v.X for v in var if "cdf_d1" in v.VarName]
cdf_d2 = [v.X for v in var if "cdf_d2" in v.VarName]
weights, deltas, P, S, d1,d2, cdf_d1, cdf_d2

([0.07064704644235043,
  0.0,
  0.0,
  0.5056701237568851,
  0.42368282980076577,
  0.0,
  0.0,
  0.0,
  0.0],
 [0.28745808290617264, 0.25262775027981643, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
 [7.91374151284656,
  7.122816693191279,
  26.879007483420303,
  7.91996275777608,
  3.088599804893649,
  17.674894697079488,
  33.402244753842695,
  13.660362428411046,
  3.2615400758318893],
 [208.27000427246094,
  271.05999755859375,
  342.32000732421875,
  422.760009765625,
  129.9199981689453,
  308.2799987792969,
  883.9600219726562,
  148.91000366210938,
  76.62999725341797],
 [0.7,
  0.9,
  2.118093279893139,
  1.0677707135226746,
  1.5897509951731363,
  1.8137697466943004,
  1.6887628583903513,
  2.633274315427822,
  2.041331226605601],
 [0.5283038113563107,
  0.7919675614148912,
  2.0436707361500757,
  1.004500238368044,
  1.549050837437826,
  1.7364005602217496,
  1.630510998731761,
  2.581762283294309,
  1.9987553741276385],
 [0.758036347776927,
  0.8159398746532406,
  0.982852249377488

In [103]:
vars = m.getVars()

var = np.array([v for v in vars if  v.VarName[:4] == "var_"])
covar = np.array([v for v in vars if  v.VarName[:6] == "covar_"])

Sigma_result = np.zeros((n, n), dtype=gp.Var)

for i in range(n):
    Sigma_result[i, i] = np.array([v.X for v in vars if v.VarName == f"var_{i}"])[0]
    for j in range(i+1, n):
        print([v for v in vars if v.VarName == f"covar_{i}_{j}"])
        Sigma_result[i, j] = np.array([v.X for v in vars if v.VarName == f"covar_{i}_{j}"])[0]
        Sigma_result[j, i] = np.array([v.X for v in vars if v.VarName == f"covar_{i}_{j}"])[0]

Sigma_result

[<gurobi.Var covar_0_1 (value 0.053521113371725784)>]
[<gurobi.Var covar_0_2 (value 0.030411673474486554)>]
[<gurobi.Var covar_0_3 (value 0.007365720158706317)>]
[<gurobi.Var covar_0_4 (value 0.004359170117730482)>]
[<gurobi.Var covar_0_5 (value 0.02739856534361188)>]
[<gurobi.Var covar_0_6 (value 0.01112200981942475)>]
[<gurobi.Var covar_0_7 (value 0.024487999704249704)>]
[<gurobi.Var covar_0_8 (value 0.007902958227140496)>]
[<gurobi.Var covar_1_2 (value 0.020083611002916404)>]
[<gurobi.Var covar_1_3 (value 0.011759594057620441)>]
[<gurobi.Var covar_1_4 (value 0.0036721745334401907)>]
[<gurobi.Var covar_1_5 (value 0.015581478962010316)>]
[<gurobi.Var covar_1_6 (value 0.0021343728591422836)>]
[<gurobi.Var covar_1_7 (value 0.013368442340672762)>]
[<gurobi.Var covar_1_8 (value 0.003773689408326417)>]
[<gurobi.Var covar_2_3 (value 0.005366696866145157)>]
[<gurobi.Var covar_2_4 (value 0.0019288074303645715)>]
[<gurobi.Var covar_2_5 (value 0.004473402421899348)>]
[<gurobi.Var covar_2_6 (val

array([[0.06368331599078264, 0.053521113371725784, 0.030411673474486554,
        0.007365720158706317, 0.004359170117730482, 0.02739856534361188,
        0.01112200981942475, 0.024487999704249704, 0.007902958227140496],
       [0.053521113371725784, 0.0258576960303869, 0.020083611002916404,
        0.011759594057620441, 0.0036721745334401907,
        0.015581478962010316, 0.0021343728591422836,
        0.013368442340672762, 0.003773689408326417],
       [0.030411673474486554, 0.020083611002916404, 0.012918098160521928,
        0.005366696866145157, 0.0019288074303645715,
        0.004473402421899348, 0.0033673806777207937,
        -0.0014226998937777865, 0.0012516425396806902],
       [0.007365720158706317, 0.011759594057620441, 0.005366696866145157,
        0.011771471365474866, 0.0018572213674529498, 0.00919030238853819,
        0.0036915367315937682, 0.010593762584972098,
        0.004085414573940683],
       [0.004359170117730482, 0.0036721745334401907,
        0.001928807430364571

In [73]:
i = 2

for D in range(1, 5):
    print(D/10)

    Phi_inv_expr = 4/1.7*( - 0.5)
    K_expr = (sigma[i]*Phi_inv_expr + mu[i] + 1) * S[i]
    d1_expr = (1 - K_expr/S[i] + T*(r + sigma[i]**2/2))/(sigma[i]*np.sqrt(T))
    d2_expr = d1_expr - sigma[i]*np.sqrt(T)

    P_expr = K_expr*np.exp(-r*T)*cdf_d2[i] - S[i]*cdf_d1[i]

    print(K_expr, S[i], P_expr,)
     #deltas[i]

#d1_expr, d2_expr, cdf_d1_expr, cdf_d2_expr

#K_expr*np.exp(-r*T)*cdf_d2_expr
S[i]

0.1
                                 Open        High         Low       Close  \
Date                                                                        
2026-04-23 00:00:00-04:00  192.486104  196.144437  190.207973  194.681099   

                                 Volume  Dividends  Stock Splits  
Date                                                              
2026-04-23 00:00:00-04:00  6.717766e+06        0.0           0.0                                    Open        High         Low       Close  \
Date                                                                        
2026-04-23 00:00:00-04:00  231.509995  235.910004  228.770004  234.149994   

                            Volume  Dividends  Stock Splits  
Date                                                         
2026-04-23 00:00:00-04:00  8079700        0.0           0.0                                   Open       High        Low      Close  \
Date                                                                    

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-04-23 00:00:00-04:00,231.509995,235.910004,228.770004,234.149994,8079700,0.0,0.0


In [14]:
(sigma[i]*Phi_inv_expr + mu[i] + 1)

np.float64(0.9531887876465414)

In [29]:
import math



CDF_pts

array([2.86651572e-07, 4.79183277e-07, 7.93328152e-07, 1.30080745e-06,
       2.11245470e-06, 3.39767312e-06, 5.41254391e-06, 8.53990547e-06,
       1.33457490e-05, 2.06575069e-05, 3.16712418e-05, 4.80963440e-05,
       7.23480439e-05, 1.07799733e-04, 1.59108590e-04, 2.32629079e-04,
       3.36929266e-04, 4.83424142e-04, 6.87137938e-04, 9.67603213e-04,
       1.34989803e-03, 1.86581330e-03, 2.55513033e-03, 3.46697380e-03,
       4.66118802e-03, 6.20966533e-03, 8.19753592e-03, 1.07241100e-02,
       1.39034475e-02, 1.78644206e-02, 2.27501319e-02, 2.87165598e-02,
       3.59303191e-02, 4.45654628e-02, 5.47992917e-02, 6.68072013e-02,
       8.07566592e-02, 9.68004846e-02, 1.15069670e-01, 1.35666061e-01,
       1.58655254e-01, 1.84060125e-01, 2.11855399e-01, 2.41963652e-01,
       2.74253118e-01, 3.08537539e-01, 3.44578258e-01, 3.82088578e-01,
       4.20740291e-01, 4.60172163e-01, 5.00000000e-01, 5.39827837e-01,
       5.79259709e-01, 6.17911422e-01, 6.55421742e-01, 6.91462461e-01,
      

In [87]:
import plotly.express as px

px.line(benchmark(all_stocks, weights, "SPY", end, dt.datetime.today()))

In [ ]:
px.line(benchmark(all_stocks, np.ones(len(all_stocks))/len(all_stocks), "SPY", end, dt.datetime.today()))

In [ ]:
sigma = [np.std(get_hist(s, start, end).Open.to_numpy()) for s in all_stocks]

sigma

In [ ]:
import plotly.express as px
import sympy as sp

K = sp.Symbol("K")
S = S[0]
sigma = sigma[0]

d1 = (1-K/S+T*(r+sigma**2/2))/(sigma*np.sqrt(T)) 

sp.plot(d1)

In [ ]:
#if m.status == GRB.INFEASIBLE:
m.computeIIS()
m.write("model.ilp")
for c in m.getConstrs():
    if c.IISConstr:
        print(f"Infeasible constraint: {c.ConstrName}")
for v in m.getVars():
    if v.IISLB:
        print(f"Infeasible lower bound: {v.VarName} lb={v.LB}")
    if v.IISUB:
        print(f"Infeasible upper bound: {v.VarName} ub={v.UB}")
    
